In [4]:
import geopandas as gpd
import pandas as pd

In [7]:
import numpy as np
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point
import odc.stac
import planetary_computer
import rioxarray
import rasterio.features
from scipy.interpolate import griddata
from scipy.ndimage import binary_dilation

# Assuming get_utm_epsg(lon, lat) is already defined in your script
def get_utm_epsg(lon, lat):
    """Dynamically calculates the correct WGS84 UTM EPSG code."""
    utm_zone = int((lon + 180) / 6) + 1
    return 32600 + utm_zone


def estimate_lake_volume_and_depth(lake_row, catalog):
    """
    Estimates the volume and max depth of a glacial lake by extrapolating
    surrounding valley topography, with empirical fallback.

    Parameters:
    - lake_row: A single row (Series) from a GeoDataFrame of lakes (EPSG:4326).
    - catalog: Initialized pystac_client for Planetary Computer.
    """
    lake_geom = lake_row.geometry

    # --- Step 1: Stream Local DEM (Small Buffer) ---
    # A 0.02 degree buffer (~2 km) is plenty to capture the immediate valley walls
    bbox = lake_geom.buffer(0.02).bounds
    search = catalog.search(collections=["cop-dem-glo-30"], bbox=bbox)
    items = list(search.items())

    if not items:
        return {
            "status": "error",
            "volume_m3": None,
            "max_depth_m": None,
            "surface_elevation_m": None,
            "area_m2": None,
            "perimeter_m": None,
            "method": "error",
            "reason": "No DEM data",
        }

    dem_dataset = odc.stac.load(
        items, bbox=bbox, crs="EPSG:4326",
        resolution=0.0002777777777777778, patch_url=planetary_computer.sign
    )

    dem_array = dem_dataset["data"].squeeze()
    if 'longitude' in dem_array.dims and 'latitude' in dem_array.dims:
        dem_array = dem_array.rename({'longitude': 'x', 'latitude': 'y'})
    dem_array = dem_array.rio.write_crs("EPSG:4326")

    # --- Step 2: Reproject to UTM (Meters) ---
    # We must calculate volume in cubic meters, so the grid must be in meters
    lon_center, lat_center = lake_geom.centroid.x, lake_geom.centroid.y
    utm_epsg = get_utm_epsg(lon_center, lat_center)
    utm_crs = f"EPSG:{utm_epsg}"

    dem_utm = dem_array.rio.reproject(utm_crs)
    lake_utm = gpd.GeoSeries([lake_geom], crs="EPSG:4326").to_crs(utm_crs).iloc[0]

    # Area and perimeter used by empirical relation and output attributes
    lake_area_m2 = float(lake_utm.area)
    lake_perimeter_m = float(lake_utm.length)

    if lake_area_m2 <= 0:
        return {
            "status": "error",
            "volume_m3": None,
            "max_depth_m": None,
            "surface_elevation_m": None,
            "area_m2": None,
            "perimeter_m": None,
            "method": "error",
            "reason": "Invalid lake geometry area",
        }

    # --- Step 3: Create Masks for Water and Shoreline ---
    out_shape = dem_utm.shape
    transform = dem_utm.rio.transform()

    # rasterize lake: 1 inside the lake, 0 outside
    lake_mask = rasterio.features.geometry_mask(
        [lake_utm], out_shape=out_shape, transform=transform, invert=True
    )

    if not lake_mask.any():
        # Empirical fallback when lake is too small for rasterized DEM workflow
        volume_m3 = 0.104 * (lake_area_m2 ** 1.42)
        mean_depth = volume_m3 / lake_area_m2
        max_depth_m = mean_depth * 1.5
        return {
            "status": "success",
            "volume_m3": round(volume_m3, 2),
            "max_depth_m": round(max_depth_m, 2),
            "surface_elevation_m": None,
            "area_m2": round(lake_area_m2, 2),
            "perimeter_m": round(lake_perimeter_m, 2),
            "method": "empirical_fallback_small_lake",
        }

    # Dilate the lake mask by ~5 pixels (150m) to grab the immediate surrounding terrain
    dilated_mask = binary_dilation(lake_mask, iterations=5)
    shore_mask = dilated_mask & ~lake_mask  # Only the land touching the lake

    # --- Step 4: Extract Coordinate Data ---
    y_coords, x_coords = np.meshgrid(dem_utm.y.values, dem_utm.x.values, indexing='ij')

    # Extract Land (shoreline) X, Y, Z
    land_points = np.column_stack((x_coords[shore_mask], y_coords[shore_mask]))
    land_z = dem_utm.values[shore_mask]

    # Extract Water X, Y (Z will be calculated)
    water_points = np.column_stack((x_coords[lake_mask], y_coords[lake_mask]))

    # If shoreline support points are insufficient, use empirical fallback
    if land_points.shape[0] < 3 or water_points.shape[0] == 0:
        volume_m3 = 0.104 * (lake_area_m2 ** 1.42)
        mean_depth = volume_m3 / lake_area_m2
        max_depth_m = mean_depth * 1.5
        return {
            "status": "success",
            "volume_m3": round(volume_m3, 2),
            "max_depth_m": round(max_depth_m, 2),
            "surface_elevation_m": None,
            "area_m2": round(lake_area_m2, 2),
            "perimeter_m": round(lake_perimeter_m, 2),
            "method": "empirical_fallback_insufficient_shore",
        }

    # --- Step 5: Interpolate the Lake Bed ---
    # We use 'cubic' to create a smooth, bowl-like basin mimicking natural erosion
    try:
        water_z_bed = griddata(land_points, land_z, water_points, method='cubic')

        # If cubic fails (returns NaNs near edges), fallback to nearest
        if np.isnan(water_z_bed).any():
            water_z_bed = griddata(land_points, land_z, water_points, method='nearest')

    except Exception as e:
        return {
            "status": "error",
            "volume_m3": None,
            "max_depth_m": None,
            "surface_elevation_m": None,
            "area_m2": round(lake_area_m2, 2),
            "perimeter_m": round(lake_perimeter_m, 2),
            "method": "error",
            "reason": f"Interpolation failed: {str(e)}",
        }

    # --- Step 6: Calculate Depth and Volume (With Empirical Fallback) ---
    # Estimate surface elevation using the 5th percentile of shoreline elevation
    surface_elevation = np.percentile(land_z, 5)

    # Depth = Surface Level - Bed Level
    depths = surface_elevation - water_z_bed
    depths = np.where(np.isfinite(depths), depths, 0)
    depths[depths < 0] = 0

    # Calculate volume: Sum of (depth * area of one pixel)
    pixel_area = abs(transform.a * transform.e)
    volume_m3 = float(np.sum(depths * pixel_area))
    max_depth_m = float(np.max(depths)) if depths.size > 0 else 0.0

    # Hybrid check: use empirical model if DEM-derived basin is unrealistic
    if volume_m3 == 0 or max_depth_m < 0.5:
        volume_m3 = 0.104 * (lake_area_m2 ** 1.42)
        mean_depth = volume_m3 / lake_area_m2
        max_depth_m = mean_depth * 1.5
        method_used = "empirical_fallback"
    else:
        method_used = "dem_extrapolation"

    return {
        "status": "success",
        "volume_m3": round(volume_m3, 2),
        "max_depth_m": round(max_depth_m, 2),
        "surface_elevation_m": round(float(surface_elevation), 2),
        "area_m2": round(lake_area_m2, 2),
        "perimeter_m": round(lake_perimeter_m, 2),
        "method": method_used,
    }

In [5]:
lakes_gdf = gpd.read_file(r'model_data\shapefiles\glacial_lakes\glake_final_18.gpkg')

In [9]:
import os
import time
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from pystac_client import Client
import planetary_computer

# Make sure estimate_lake_volume_and_depth and get_utm_epsg are defined above this

def open_catalog_with_retry(max_retries=6, base_delay=5):
    """Open Planetary Computer STAC with retry/backoff for transient DNS/network issues."""
    last_err = None
    for attempt in range(max_retries):
        try:
            return Client.open(
                "https://planetarycomputer.microsoft.com/api/stac/v1",
                modifier=planetary_computer.sign_inplace
            )
        except Exception as e:
            last_err = e
            msg = str(e)
            transient = any(x in msg for x in [
                "NameResolutionError", "getaddrinfo failed", "Temporary failure in name resolution",
                "HTTPSConnectionPool", "Max retries exceeded", "timeout", "429", "502", "503", "Connection"
            ])
            if transient and attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                print(f"Catalog connect failed (attempt {attempt + 1}/{max_retries}): {msg}")
                print(f"Retrying in {delay}s...")
                time.sleep(delay)
            else:
                raise RuntimeError(f"Could not open STAC catalog after {attempt + 1} attempt(s): {msg}") from e
    raise RuntimeError(f"Could not open STAC catalog: {last_err}")


def process_lake_volumes(lakes_gdf, output_csv, max_retries=5, base_delay=5):
    """
    Processes lakes to estimate volume and depth, featuring automatic
    checkpointing and an exponential backoff retry loop for API stability.
    Also writes area/perimeter in lakes_gdf and CSV.
    """
    # Ensure output columns exist in the input GeoDataFrame
    if 'area' not in lakes_gdf.columns:
        lakes_gdf['area'] = np.nan
    if 'perimeter' not in lakes_gdf.columns:
        lakes_gdf['perimeter'] = np.nan

    # 1. Initialize STAC Client once (with retry)
    catalog = open_catalog_with_retry(max_retries=max_retries, base_delay=base_delay)

    # 2. Load checkpoints to resume if stopped
    processed_ids = set()
    if os.path.exists(output_csv):
        try:
            existing_df = pd.read_csv(output_csv)
            id_col = 'GLAKE_ID' if 'GLAKE_ID' in existing_df.columns else ('lake_id' if 'lake_id' in existing_df.columns else None)
            if id_col is not None:
                processed_ids = set(existing_df[id_col].dropna().astype(str))
                print(f"Resuming from checkpoint: {len(processed_ids)} lakes already processed.")
        except pd.errors.EmptyDataError:
            pass

    write_header = not os.path.exists(output_csv)

    # 3. Open output file in append mode
    with open(output_csv, 'a') as f:
        if write_header:
            f.write("GLAKE_ID,status,volume_m3,max_depth_m,surface_elevation_m,area_m2,perimeter_m,method,error_reason\n")

        for index, lake_row in tqdm(lakes_gdf.iterrows(), total=len(lakes_gdf), desc="Estimating Volumes"):
            # Use GLAKE_ID as requested; fallback if missing
            glake_id = str(lake_row.get('GLAKE_ID', lake_row.get('lake_id', index)))

            if glake_id in processed_ids:
                continue

            # --- Exponential Backoff Retry Loop ---
            success = False
            for attempt in range(max_retries):
                try:
                    # Call the morphometry pipeline
                    result = estimate_lake_volume_and_depth(lake_row, catalog)

                    status = result.get('status', 'unknown')
                    vol = result.get('volume_m3', '')
                    depth = result.get('max_depth_m', '')
                    elev = result.get('surface_elevation_m', '')
                    area_m2 = result.get('area_m2', '')
                    perimeter_m = result.get('perimeter_m', '')
                    method = result.get('method', '')
                    reason = result.get('reason', '')

                    # Update lakes_gdf columns in-place
                    if area_m2 not in ['', None]:
                        lakes_gdf.at[index, 'area'] = area_m2
                    if perimeter_m not in ['', None]:
                        lakes_gdf.at[index, 'perimeter'] = perimeter_m

                    f.write(f"{glake_id},{status},{vol},{depth},{elev},{area_m2},{perimeter_m},{method},{reason}\n")
                    f.flush()
                    success = True
                    break  # Exit retry loop on success

                except Exception as e:
                    error_msg = str(e)
                    # Handle API/Network timeouts
                    if any(x in error_msg for x in ["429", "502", "503", "timeout", "Connection", "NameResolutionError", "getaddrinfo failed"]):
                        delay = base_delay * (2 ** attempt)
                        tqdm.write(f"Network error on lake {glake_id}. Retrying in {delay}s...")
                        time.sleep(delay)
                    else:
                        # Structural error, log and move on
                        f.write(f"{glake_id},error,,,,,,,{error_msg.replace(',', ';')}\n")
                        f.flush()
                        success = True
                        break

            if not success:
                f.write(f"{glake_id},error,,,,,,,Max API retries exceeded\n")
                f.flush()


# --- Execution ---
# IMPORTANT: Pre-project to geographic coordinates to prevent bounding box errors
lakes_gdf = lakes_gdf.to_crs("EPSG:4326")
process_lake_volumes(lakes_gdf, "hma_lake_volume_depth.csv")

Estimating Volumes:  94%|█████████▎| 6474/6923 [1:15:03<01:59,  3.77it/s]  

Network error on lake GL099611E30377N. Retrying in 5s...


Estimating Volumes:  94%|█████████▎| 6474/6923 [1:15:08<01:59,  3.77it/s]

Network error on lake GL099611E30377N. Retrying in 10s...


Estimating Volumes: 100%|██████████| 6923/6923 [1:20:39<00:00,  1.43it/s]  


In [6]:
import pandas as pd

# 1) Read CSV
vol_df = pd.read_csv("hma_lake_volume_depth.csv")

# 2) Keep only required columns
keep_cols = ["GLAKE_ID", "volume_m3", "max_depth_m", "surface_elevation_m", "area_m2", "perimeter_m"]
vol_df = vol_df[keep_cols].copy()

# 3) Clean key types for safe join
lakes_gdf["GLAKE_ID"] = lakes_gdf["GLAKE_ID"].astype(str).str.strip()
vol_df["GLAKE_ID"] = vol_df["GLAKE_ID"].astype(str).str.strip()

# 4) Optional: if CSV has duplicate IDs, keep first occurrence
vol_df = vol_df.drop_duplicates(subset=["GLAKE_ID"], keep="first")

# 5) Merge into lakes_gdf
lakes_gdf = lakes_gdf.merge(vol_df, on="GLAKE_ID", how="left")

print("Merge complete.")
print(lakes_gdf[["GLAKE_ID", "volume_m3", "max_depth_m", "surface_elevation_m", "area_m2", "perimeter_m"]].head())

Merge complete.
          GLAKE_ID  volume_m3  max_depth_m  surface_elevation_m    area_m2  \
0  GL077099E35395N  107025.04         9.41              4551.11   68538.18   
1  GL078223E34278N  230093.88        17.34              5251.23   58467.51   
2  GL078079E34401N  103637.37         4.27              5309.85  224056.20   
3  GL078069E34414N   48498.13         7.18              5379.77   74200.97   
4  GL078100E34520N  575485.42        18.09              5269.30  170130.16   

   perimeter_m  
0      1201.83  
1      1319.63  
2      2939.71  
3      1074.10  
4      1835.04  


In [1]:
print('hello')

hello


In [7]:
lakes_gdf.head()

,Type,Elevation,GLAKE_ID,area_2020,perimeter_2020,area_1990,perimeter_1990,geometry_1990,area_2000,perimeter_2000,...,freeboard_m_dam,wse_m_dam,eq_susceptibility,ls_susceptibility,geometry,volume_m3,max_depth_m,surface_elevation_m,area_m2,perimeter_m
0,Pro-glacial lakes,4556.0,GL077099E35395N,68531.572797,1236.586129,106050.004106,1314.532748,"POLYGON ((-1484718.293773 782310.478045, -1484...",113754.339222,1449.811508,...,17.950400706922665,4551.557681051051,0.273678,5.0,"MULTIPOLYGON (((-1484700.074 782331.377, -1484...",107025.04,9.41,4551.11,68538.18,1201.83
1,Pro-glacial lakes,5257.0,GL078223E34278N,58419.996563,1303.817185,38556.970725,807.861427,"POLYGON ((-1415403.92275 633222.731632, -14154...",72417.311662,1311.410644,...,33.80811544502285,5250.6051017949985,0.192478,4.0,"MULTIPOLYGON (((-1415347.685 633371.82, -14153...",230093.88,17.34,5251.23,58467.51,1319.63
2,Pro-glacial lakes,5313.0,GL078079E34401N,223837.087227,2963.350846,129659.049210,1941.270084,"POLYGON ((-1425011.936687 649533.145087, -1425...",139594.561238,1849.796900,...,4.688485788738035,5309.854989327003,0.197459,3.0,"MULTIPOLYGON (((-1424755.132 650202.033, -1424...",103637.37,4.27,5309.85,224056.20,2939.71
3,Pro-glacial lakes,5392.0,GL078069E34414N,74127.612413,1046.494224,NaN,NaN,None,22144.984899,561.758122,...,17.340527144654516,5383.332595685207,0.197459,4.0,"MULTIPOLYGON (((-1425484.421 651570.284, -1425...",48498.13,7.18,5379.77,74200.97,1074.10
4,Pro-glacial lakes,5272.0,GL078100E34520N,169968.977931,1796.160873,38415.044338,955.956017,"POLYGON ((-1420591.675208 663605.76101, -14206...",43785.982956,931.446951,...,16.07811092391148,5268.944359978513,0.205140,4.0,"MULTIPOLYGON (((-1420641.234 663520.885, -1420...",575485.42,18.09,5269.30,170130.16,1835.04


In [8]:
output_path = r'model_data\shapefiles\glacial_lakes\glake_final_19.gpkg'
lakes_gdf.to_file(output_path, driver="GPKG")

In [ ]:
glake = gpd.read_file(r'model_data\shapefiles\glacial_lakes\glake_final_19.gpkg')
glake.columns

Index(['Type', 'Elevation', 'GLAKE_ID', 'area_2020', 'perimeter_2020',
       'area_1990', 'perimeter_1990', 'geometry_1990', 'area_2000',
       'perimeter_2000', 'geometry_2000', 'area_2010', 'perimeter_2010',
       'geometry_2010', 'area_2015', 'perimeter_2015', 'geometry_2015',
       'area_1990_km2', 'area_2000_km2', 'area_2010_km2', 'area_2015_km2',
       'area_2020_km2', 'expansion_rate_km2_yr', 'expansion_rate_pct_yr',
       'glof_happened', 'glof_count', 'lake_type', 'distance_to_glacier_m',
       'nearest_rgiid', 'is_connected', 'dam_crest_m', 'dam_height_m',
       'dam_slope_deg', 'dam_width_m', 'freeboard_m', 'wse_m',
       'dam_crest_m_dam', 'dam_height_m_dam', 'dam_slope_deg_dam',
       'dam_width_m_dam', 'freeboard_m_dam', 'wse_m_dam', 'eq_susceptibility',
       'ls_susceptibility', 'volume_m3', 'max_depth_m', 'surface_elevation_m',
       'area_m2', 'perimeter_m', 'geometry'],
      dtype='object')

In [10]:
columns = ['dam_crest_m', 'dam_height_m', 'dam_slope_deg', 'dam_width_m', 'freeboard_m', 'wse_m']
glake.drop(columns, errors='ignore', inplace=True)
glake.head()

,Type,Elevation,GLAKE_ID,area_2020,perimeter_2020,area_1990,perimeter_1990,geometry_1990,area_2000,perimeter_2000,...,freeboard_m_dam,wse_m_dam,eq_susceptibility,ls_susceptibility,volume_m3,max_depth_m,surface_elevation_m,area_m2,perimeter_m,geometry
0,Pro-glacial lakes,4556.0,GL077099E35395N,68531.572797,1236.586129,106050.004106,1314.532748,"POLYGON ((-1484718.293773 782310.478045, -1484...",113754.339222,1449.811508,...,17.950400706922665,4551.557681051051,0.273678,5.0,107025.04,9.41,4551.11,68538.18,1201.83,"MULTIPOLYGON (((-1484700.074 782331.377, -1484..."
1,Pro-glacial lakes,5257.0,GL078223E34278N,58419.996563,1303.817185,38556.970725,807.861427,"POLYGON ((-1415403.92275 633222.731632, -14154...",72417.311662,1311.410644,...,33.80811544502285,5250.6051017949985,0.192478,4.0,230093.88,17.34,5251.23,58467.51,1319.63,"MULTIPOLYGON (((-1415347.685 633371.82, -14153..."
2,Pro-glacial lakes,5313.0,GL078079E34401N,223837.087227,2963.350846,129659.049210,1941.270084,"POLYGON ((-1425011.936687 649533.145087, -1425...",139594.561238,1849.796900,...,4.688485788738035,5309.854989327003,0.197459,3.0,103637.37,4.27,5309.85,224056.20,2939.71,"MULTIPOLYGON (((-1424755.132 650202.033, -1424..."
3,Pro-glacial lakes,5392.0,GL078069E34414N,74127.612413,1046.494224,NaN,NaN,None,22144.984899,561.758122,...,17.340527144654516,5383.332595685207,0.197459,4.0,48498.13,7.18,5379.77,74200.97,1074.10,"MULTIPOLYGON (((-1425484.421 651570.284, -1425..."
4,Pro-glacial lakes,5272.0,GL078100E34520N,169968.977931,1796.160873,38415.044338,955.956017,"POLYGON ((-1420591.675208 663605.76101, -14206...",43785.982956,931.446951,...,16.07811092391148,5268.944359978513,0.205140,4.0,575485.42,18.09,5269.30,170130.16,1835.04,"MULTIPOLYGON (((-1420641.234 663520.885, -1420..."
